# Bitcoin / Chronos

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `BTC_Chronos.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
############# Amazon Chronos (Foundation Model)

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from binance.client import Client

import torch
from chronos import ChronosPipeline
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

print(f"Torch version: {torch.__version__}")
print("Amazon Chronos library loaded.")

In [ ]:
# ========== FETCH BTC LOW PRICE FROM BINANCE (WITH CACHE & AUTO-UPDATE) ==========
# Cache raw daily lows to avoid re-hitting the API on each run
DATA_CACHE = Path("../../data/raw/btc_low_daily.csv")

def fetch_btc_low_binance(start: str, end: str, max_retries: int = 3) -> pd.DataFrame:
    """
    Download Bitcoin OHLC data from the Binance API and keep only the LOW column.
    Returns a DataFrame with columns ['ds', 'y'].
    """
    import time

    client = Client()

    for attempt in range(max_retries):
        try:
            print(f"[INFO] Fetching data from Binance API ({start} -> {end}) (attempt {attempt + 1}/{max_retries})...")

            klines = client.get_historical_klines(
                "BTCUSDT",
                Client.KLINE_INTERVAL_1DAY,
                start,
                end
            )

            if not klines:
                # It's possible there's no new data yet if we are too close to the last close
                print("[WARNING] Binance returned empty data (maybe up to date?).")
                return pd.DataFrame(columns=["ds", "y"])

            df_raw = pd.DataFrame(klines, columns=[
                "timestamp", "open", "high", "low", "close",
                "volume", "close_time", "quote_asset_volume",
                "number_of_trades", "taker_buy_base_asset_volume",
                "taker_buy_quote_asset_volume", "ignore"
            ])

            df_raw["ds"] = pd.to_datetime(df_raw["timestamp"], unit="ms")
            df_raw["y"] = pd.to_numeric(df_raw["low"], errors="coerce")

            df_clean = (
                df_raw.dropna(subset=["ds", "y"])
                .loc[df_raw["y"] > 0, ["ds", "y"]]
                .sort_values("ds")
                .reset_index(drop=True)
            )

            print(f"[INFO] ✅ Downloaded {len(df_clean)} rows from Binance.")
            if not df_clean.empty:
                print(f"[INFO] Coverage: {df_clean['ds'].min().date()} → {df_clean['ds'].max().date()}")
            return df_clean

        except Exception as exc:
            error_msg = str(exc).lower()
            if "rate limit" in error_msg or "429" in error_msg or "1003" in error_msg:
                wait_time = (attempt + 1) * 10
                print(f"[WARNING] Rate limit hit. Waiting {wait_time}s before retrying...")
                time.sleep(wait_time)
            else:
                print(f"[ERROR] {exc}")
                if attempt == max_retries - 1:
                    raise
                time.sleep(5)

    raise RuntimeError(f"Unable to download data after {max_retries} attempts.")

START = "2017-08-17"
END = datetime.utcnow().strftime("%Y-%m-%d")

if DATA_CACHE.exists():
    df = pd.read_csv(DATA_CACHE, parse_dates=["ds"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["ds", "y"]).sort_values("ds").reset_index(drop=True)
    
    last_date = df["ds"].max()
    # If cache is older than yesterday, try to update
    if last_date < pd.to_datetime(END) - timedelta(days=1):
        print(f"[INFO] Cache outdated (Last date: {last_date.date()}). Fetching update...")
        new_start = (last_date + timedelta(days=1)).strftime("%Y-%m-%d")
        try:
            df_new = fetch_btc_low_binance(new_start, END)
            if not df_new.empty:
                df = pd.concat([df, df_new], ignore_index=True)
                df = df.drop_duplicates(subset="ds").sort_values("ds").reset_index(drop=True)
                df.to_csv(DATA_CACHE, index=False)
                print(f"[INFO] Updated cache with {len(df_new)} new rows.")
            else:
                print("[INFO] No new data found.")
        except Exception as e:
             print(f"[WARNING] Could not update data: {e}")
    else:
        print(f"[INFO] Loaded {len(df)} rows from cache {DATA_CACHE} (Up to date).")
else:
    df = fetch_btc_low_binance(START, END)
    df.to_csv(DATA_CACHE, index=False)
    print(f"[INFO] Saved fresh download to {DATA_CACHE.resolve()}")

print(f"\n[INFO] Data source: Binance API (BTCUSDT LOW) | Rows: {len(df)}")
print(f"[INFO] Stats → Min={df['y'].min():.2f}, Max={df['y'].max():.2f}, Mean={df['y'].mean():.2f}")
df.head(10)

In [ ]:
# ========== DATA PREPROCESSING FOR TIMESFM ==========
# TimesFM is a foundation model that doesn't require feature engineering
# It works directly with univariate time series

def preprocess_data(price_df: pd.DataFrame) -> pd.DataFrame:
    """Prepare data for TimesFM - simple preprocessing."""
    df_clean = (
        price_df
        .drop_duplicates(subset="ds")
        .sort_values("ds")
        .reset_index(drop=True)
    )

    # Ensure a continuous daily index and forward-fill gaps
    df_clean = (
        df_clean.set_index("ds")
        .resample("D")
        .ffill()
        .reset_index()
    )
    
    df_clean["y"] = pd.to_numeric(df_clean["y"], errors="coerce")
    df_clean = df_clean.dropna(subset=["y"]).reset_index(drop=True)
    
    return df_clean[["ds", "y"]]

df = preprocess_data(df)
print(f"[INFO] Preprocessed data shape: {df.shape}")
print(f"[INFO] Date range: {df['ds'].min().date()} → {df['ds'].max().date()}")
df.head(10)

In [ ]:
# Basic statistics of the time series
df[["y"]].describe().transpose()

In [ ]:
# Check for missing values
print(f"Missing values in 'y': {df['y'].isna().sum()}")
print(f"Missing values in 'ds': {df['ds'].isna().sum()}")

In [ ]:
# ========== TRAIN / VALIDATION SPLIT CONFIGURATION ==========
VAL_HORIZON = 180  # days kept aside for validation
FORECAST_HORIZON = 60  # days to forecast ahead

if len(df) <= VAL_HORIZON:
    raise ValueError("Validation horizon is larger than the dataset length.")

df_train = df.iloc[:-VAL_HORIZON].reset_index(drop=True)
df_val = df.iloc[-VAL_HORIZON:].reset_index(drop=True)

print(f"[INFO] Training rows: {len(df_train)}")
print(f"[INFO] Validation rows: {len(df_val)}")
print(f"[INFO] Forecast horizon: {FORECAST_HORIZON} days")

In [ ]:
# ========== LOAD AMAZON CHRONOS MODEL ==========
# Chronos is a family of pretrained time series forecasting models based on language model architectures.
# It tokenizes time series values and treats forecasting as a next-token prediction task.

# Initialize Chronos Pipeline
# Using "amazon/chronos-t5-large" (approx 710M params)
# Other options: "amazon/chronos-t5-tiny", "amazon/chronos-t5-small", "amazon/chronos-t5-base"

print("[INFO] Loading Amazon Chronos (t5-large) model...")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-large",
    device_map="cpu",  # Use "cuda" if you have a GPU
    torch_dtype=torch.float32,
)

print("[INFO] Amazon Chronos model loaded successfully!")
print(f"[INFO] Model: amazon/chronos-t5-large (710M parameters)")
print(f"[INFO] Forecast horizon: {FORECAST_HORIZON} days")

In [ ]:
# ========== VALIDATION METRICS ==========
# Use training data to forecast validation period

# Prepare input for Chronos
# Chronos expects a torch tensor of shape (batch_size, sequence_length)
context_tensor = torch.tensor(df_train["y"].values, dtype=torch.float32)

# Generate forecast for validation period with progress tracking
# num_samples: number of probabilistic samples to generate
print("[INFO] Generating validation forecasts...")
from tqdm import tqdm
import sys

forecast_samples = pipeline.predict(
    inputs=context_tensor,
    prediction_length=VAL_HORIZON,
    num_samples=20,
)
print("[INFO] ✅ Validation forecast complete")

# forecast_samples shape: (batch_size, num_samples, prediction_length)
# We take the median as the point forecast
val_preds = np.median(forecast_samples[0].numpy(), axis=0)
val_actual = df_val["y"].values[:len(val_preds)]

# Create results dataframe
val_results = pd.DataFrame({
    "ds": df_val["ds"].iloc[:len(val_preds)],
    "y": val_actual,
    "yhat": val_preds
})

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(y_true - y_pred))))

def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.where(y_true == 0, np.nan, np.abs(y_true))
    return float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)

val_mae = float(np.mean(np.abs(val_results["y"] - val_results["yhat"])))
val_rmse = rmse(val_results["y"].to_numpy(), val_results["yhat"].to_numpy())
val_mape = mape(val_results["y"].to_numpy(), val_results["yhat"].to_numpy())

direction_df = val_results[["y", "yhat"]].diff().dropna()
if direction_df.empty:
    val_direction = float("nan")
else:
    val_direction = float(np.mean(np.sign(direction_df["y"]) == np.sign(direction_df["yhat"])))

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS (Chronos t5-large)")
print(f"{'='*50}")
print(f"MAE:                  {val_mae:>12,.2f}")
print(f"RMSE:                 {val_rmse:>12,.2f}")
print(f"MAPE:                 {val_mape:>12,.2f}%")
if np.isnan(val_direction):
    print(f"Directional accuracy: {'N/A':>12}")
else:
    print(f"Directional accuracy: {val_direction * 100:>12.2f}%")
print(f"{'='*50}\n")

print(f"[INFO] Last {min(FORECAST_HORIZON, len(val_results))} validation points:")
val_results.tail(FORECAST_HORIZON)

In [ ]:
# ========== GENERATE FORECAST ==========
# Use full dataset to forecast future

# Prepare input for Chronos (full history)
full_context_tensor = torch.tensor(df["y"].values, dtype=torch.float32)

# Generate forecast with progress indication
print("[INFO] Generating 60-day future forecast (this may take a few moments)...")
print("[INFO] Model: amazon/chronos-t5-large | Processing...")

forecast_samples_future = pipeline.predict(
    inputs=full_context_tensor,
    prediction_length=FORECAST_HORIZON,
    num_samples=50,  # More samples for better distribution
)

print("[INFO] ✅ Forecast generation complete\n")

# Extract median and quantiles
forecast_median = np.median(forecast_samples_future[0].numpy(), axis=0)
forecast_lower = np.quantile(forecast_samples_future[0].numpy(), 0.1, axis=0)
forecast_upper = np.quantile(forecast_samples_future[0].numpy(), 0.9, axis=0)

# Create future dates
last_date = df["ds"].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=FORECAST_HORIZON, freq="D")

# Create forecast dataframe
forecast_future = pd.DataFrame({
    "ds": future_dates,
    "yhat": forecast_median,
    "yhat_lower": forecast_lower,
    "yhat_upper": forecast_upper
})

# Combine history with forecast for plotting
forecast = pd.concat([
    df[["ds", "y"]].assign(yhat=np.nan, yhat_lower=np.nan, yhat_upper=np.nan),
    forecast_future.assign(y=np.nan)
], ignore_index=True)

print(f"[INFO] Generated {FORECAST_HORIZON}-day forecast")
print(f"[INFO] Forecast range: {forecast_future['ds'].min().date()} → {forecast_future['ds'].max().date()}\n")
forecast_future.tail(FORECAST_HORIZON)

In [ ]:
# ========== SIMPLE FORECAST PLOT ==========
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))

# Plot historical data
ax.plot(df["ds"], df["y"], label="Historical", color="#1f77b4")

# Plot forecast
ax.plot(forecast_future["ds"], forecast_future["yhat"], label="Chronos Forecast", color="#ff7f0e", linewidth=2)
ax.fill_between(forecast_future["ds"], forecast_future["yhat_lower"], forecast_future["yhat_upper"], color="#ff7f0e", alpha=0.2, label="80% Confidence Interval")

ax.axvline(last_date, color="black", linestyle="--", alpha=0.7, label="Forecast start")
ax.set_title("Bitcoin LOW Price - Amazon Chronos Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ========== DETAILED FORECAST PLOT (Last N Days + Forecast) ==========
import matplotlib.pyplot as plt

history_window = 240

fig, ax = plt.subplots(figsize=(12, 6))

# Plot recent history
recent_history = df.tail(history_window)
ax.plot(recent_history["ds"], recent_history["y"], label="Actual", color="#1f77b4")

# Plot forecast
ax.plot(forecast_future["ds"], forecast_future["yhat"], label="Chronos Forecast", color="#ff7f0e", linewidth=2)
ax.fill_between(forecast_future["ds"], forecast_future["yhat_lower"], forecast_future["yhat_upper"], color="#ff7f0e", alpha=0.2, label="80% Confidence Interval")

ax.axvline(last_date, color="black", linestyle="--", alpha=0.7, label="Forecast start")
ax.set_title("Bitcoin LOW Price Forecast (History vs Forecast)")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ========== 7-DAY FORECAST SNAPSHOT ==========
next7 = forecast_future.head(7).copy()
next7["pct_change"] = next7["yhat"].pct_change().fillna(0.0) * 100

print("\nDự đoán cho 7 ngày tới (Amazon Chronos):")
print(next7.reset_index(drop=True))

print("\nChi tiết từng ngày:")
for _, row in next7.iterrows():
    date = row["ds"].date()
    value = row["yhat"]
    lower = row["yhat_lower"]
    upper = row["yhat_upper"]
    pct = row["pct_change"]
    print(f"{date}: {value:,.2f} USD ({pct:+.2f}%) | [{lower:,.2f}, {upper:,.2f}]")